In [ ]:
import pandas as pd,glob
import numpy as np
from collections import Counter


In [ ]:
df = pd.read_csv("../eabl parsed csv/sales_products_merged_cleaned.csv")
df["quantity_cases"] = df["quantity"] * df["Cases_Conversion"]
df["returns_cases"] = df["return_quantity"] * df["Cases_Conversion"]
df

In [ ]:
# Just the column names
print(list(df.columns))

In [ ]:
# Column names + data types + non-null counts
print(df.info())


In [ ]:
# Column names + data types, null counts, and unique value counts together
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "unique_values": df.nunique()
})
print(summary)

In [ ]:
# Are they identical everywhere?
are_identical = (df["segment"] == df["segment_grouped"]).all()
print("Identical in every row?", are_identical)

# How many rows differ?
diff_count = (df["segment"] != df["segment_grouped"]).sum()
print("Rows where they differ:", diff_count)
print("Rows where they match:", len(df) - diff_count)

# See what the differences actually look like
diff_rows = df[df["segment"] != df["segment_grouped"]][["segment", "segment_grouped"]].drop_duplicates()
print(diff_rows)


### What is the finest level of detail your data is captured at, and is that level actually usable for building a model — or is it too sparse/noisy to model directly?

### Is there a trend in a specific customer purchasing eg Tusker in terms of quantity in different days of the week so that we can forecast what they are likely to but when they show up.

#### How many times a combination of "customer_code", "Product_Code", "visit_date" shows up

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
grp = df.groupby(["customer_code", "Product_Code", "visit_date"]).size()
grp

#### Counts how many buckets exist in total (i.e., how many distinct customer+product+day combinations show up at all).

In [ ]:
n_total = len(grp)
n_total

In [ ]:
#Counts how many buckets exist in total (i.e., how many distinct customer+product+day combinations show up at all).
pct_singleton = (grp == 1).mean() * 100
print(n_total, pct_singleton)


### Fixing of the CustomerMaster.xlsx dataset by removing useless Columns

In [ ]:
cm = pd.read_excel("../EABL dataset/CustomerMaster_cleanV1.xlsx")

cm

In [ ]:
cm.isna().sum()

In [ ]:
columns_to_drop = [
    "days_since_last_login",
    "GPS Co-ordinate",
    "RSS Last Login Date",
    "RSS Date Registed",
    "RSS First Login Date",
    "has_alternate_name",
    "price_group_missing",
    "rss_date_filled",
    "rss_date_unresolved",
    "registration_batch",
    "ever_logged_in",
    "login_status",
    "gps_missing",
    "gps_address_matched",
    "gps_unresolved",
]


# errors="ignore" means it won't crash if a name doesn't match exactly
cm_trimmed = cm.drop(columns=columns_to_drop, errors="ignore")

# Save to a NEW file — CustomerMaster.xlsx itself is never touched
cm_trimmed.to_excel("../EABL dataset/CustomerMaster_clean_forecast.xlsx", index=False)

print("Original columns:", len(cm.columns))
print("Trimmed columns:", len(cm_trimmed.columns))
print(list(cm_trimmed.columns))
cm_trimmed

In [ ]:
cm_trimmed.isna().sum()

### "Does my data actually have layers/Hierachy — like a family tree — or is everything just at one flat level pretending to have layers?"



In [ ]:
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")     # columns: Category2, Brand, Product_Code

for c in ["Category2", "Brand"]:
#how many different Categories exist
#how many different Brands exist
    print(c, pm[c].nunique())
print("SKU", pm["Product_Code"].nunique())
for c in ["Division", "Area", "Territory", "Route"]:
    print(c, cm_trimmed[c].nunique())


### Determine if :
#### i) Using sales_products_merged_cleaned.csv over fact_table.csv was a right decsion
#### ii) If the Category and Category2  column are the same in the product_master_cleaned.csv dataset
#### iii) If the segment and segment_grouped  column are the same in the product_master_cleaned.csv dataset

In [ ]:
pm.isna().sum()

**Practical action:**

- Use `segment_grouped` as the default hierarchy level, since it offers more stability.
- If Restaurant/Supermarket-specific decisions ever matter to the business, keep the raw `segment` column available as a secondary cut to drill into.
- Don't discard the raw `segment` column entirely just because `segment_grouped` is the main tree.

In [ ]:
fact = pd.read_csv("../eabl parsed csv/fact_table.csv")                          # columns: Brand

pct_unmatched = fact["Brand"].isna().mean() * 100
pct_cat_diff = (pm["Category"] != pm["Category2"]).mean() * 100
pct_seg_diff = (df["segment"] != df["segment_grouped"]).mean() * 100
print(pct_unmatched, pct_cat_diff, pct_seg_diff)


## Does Any of My Dataset Contain "holiday", "payday", "pay day", "festive", "event"]

In [ ]:
keywords = ["holiday", "payday", "pay day", "festive", "event"]
for f in glob.glob("**/*.csv", recursive=True) + glob.glob("**/*.xlsx", recursive=True):
    try:
        cols = (pd.read_csv(f, nrows=0) if f.endswith(".csv") else pd.read_excel(f, nrows=0)).columns
    except Exception:
        continue
    hits = [c for c in cols if any(k in str(c).lower() for k in keywords)]
    if hits:
        print(f, hits)


## Seasonal Adjustment — Confidence by Period

**July–February (including December):**

- Any seasonal adjustment used for these months should be treated as an **assumption or industry benchmark**, not something proven by our own data.
- Don't let a model "discover" a seasonal pattern from our own data for these months — there's only one observation each, so any pattern found could just as easily be a one-off event as a real trend.
- Instead:
  - Use a conservative, documented assumption (e.g., an industry-standard "festive season uplift %" for beverage/alcohol distribution, or input from EABL's own commercial/sales team, who already have a sense of how demand shifts around Christmas).
  - Label it clearly as an **assumption**, not a fitted result — so nobody later mistakes it for something the data proved.

**March–June:**

- Can be treated with slightly more (though still limited) confidence.
- We can cautiously use our own data's pattern, but shouldn't treat it as fully confirmed — 2 occurrences is barely above the minimum to even compare, not enough to rule out coincidence.
- Use it, but keep it as a **soft signal** rather than a hard-coded rule.
### Using the March–June window to find holidays 

- Within the March–June window, several real Kenyan public holidays fall in **both 2025 and 2026** — so for these specific dates, there are genuinely 2 occurrences to compare, not just a vague "the whole month repeated" argument.

| Holiday | 2025 date | 2026 date |
|---|---|---|
| Good Friday | March 28 | April 3 |
| Easter Monday | March 31 | April 6 |
| Labour Day | May 1 | May 1 |
| Madaraka Day | June 1 | June 1 |

- Since each of these falls within the covered data range in both years, it's possible to check: did sales spike around Easter 2025 the same way they did around Easter 2026?
- That's a real, if modest, test — not just an assumption.

### How to apply it to a brand/region

- This connects to something already confirmed: the data has enough density at the Category × Segment intersection (only 1.1% of cells were thin) to detect uneven effects.
- So a reasonable question becomes: *"Did Spirits sales in Nightclubs spike more than Beer sales in Wholesalers around Labour Day, in both 2025 and 2026?"*
- That's a legitimate, testable question with the current data.

### Two things to do before actually running this

- **Build the calendar.** No holiday-marking table exists anywhere in the project — a small lookup needs to be created manually with these specific dates (plus any others relevant to March–June) and joined onto the date column.
- **Keep the confidence level honest.** Even with 2 occurrences, this is still "weak but usable" evidence, not proof — good enough to say *"this looks like a real pattern worth watching,"* not strong enough to bet a large inventory decision on without also applying business judgment.

> Good instinct — this is a much sharper way to use the March–June advantage than just treating the whole 4-month block as one vague area of "slightly more confidence."

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
months = sorted(df["visit_date"].dt.to_period("M").astype(str).unique())
name_counts = Counter(pd.Period(m).strftime("%B") for m in months)
print(months)
print(name_counts)


In [ ]:
df.shape

In [ ]:
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")[["Product_Code", "Category2"]]

# join Category2 onto the sales data, matching on Product_Code
df = df.merge(pm, on="Product_Code", how="left")

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["month"] = df["visit_date"].dt.to_period("M")

cells = df.groupby(["Category2", "segment_grouped", "month"]).size()
print(len(cells), (cells < 30).mean() * 100, (cells < 5).mean() * 100)
print(cells.describe())

df.to_csv("../eabl parsed csv/sales_products_merged_cleaned_2newcategories.csv", index=False)
print(df.shape)

In [ ]:
df.shape

In [ ]:
# Columns we agreed are not needed for forecasting
columns_to_drop = [
    "source_file",
    "salesperson",
    "cust_type",
    "Product_Description",
    "Handheld_Description",
    "Default_UOM",
    "Pack_Format",
    "Bottles_Per_Case",
    "Pack_Size_mls",
    'FY25_July_NSV_Conversion_1st', 
    'FY25_November_NSV_Conversion_1st', 
    'FY25_January_NSV_Conversion_1st', 
    'FY26_October_NSV_Conversion_18th', 
    'FY25_EX_KBL_Price_Per_Case', 
    'FY26_October_NSV_Conversion_18th_Updated', 
    'FY26_June_NSV_Conversion_15th', 'FY26_EX_KBL_Price_Per_Case'
]

# drop() with a new variable name leaves 'df' (and the original CSV) untouched
df_trimmed = df.drop(columns=columns_to_drop)

# Save to a NEW file — original sales_products_merged_cleaned.csv is never overwritten
df_trimmed.to_csv("../eabl parsed csv/sales_products_forecast_ready.csv", index=False)

print("Original columns:", len(df.columns))
print("Trimmed columns:", len(df_trimmed.columns))
print(list(df_trimmed.columns))


In [ ]:
df_trimmed.shape

In [ ]:
df_trimmed.isna().sum()

In [ ]:
df_trimmed